## Step 1: Setup

| Property | Value |
|---|---|
| Origin | LangGraph reference agentic-RAG pattern, formalised by Jeong et al. (Adaptive-RAG, 2024) |

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# !pip install langchain langchain-google-genai langchain-community chromadb

## Step 2: Create Your Knowledge Base

We'll use custom documents about a fictional cafe, **Cafe Lumiere**:

In [ ]:
from langchain.schema import Document

# Defining fictional data about the cafe
docs = [
    Document(
        page_content="Cafe Lumiere was founded by Elise Moreau, a Paris-trained barista passionate about sustainable coffee sourcing.",
        metadata={"source": "about.txt"},
    ),
    Document(
        page_content="The cafe offers espresso drinks between $3 to $5, pour-overs at $6, and pastries ranging from $2.5 to $4.",
        metadata={"source": "menu.txt"},
    ),
    Document(
        page_content="Open from Tuesday to Sunday. Weekdays: 8 AM to 6 PM. Weekends: 9 AM to 8 PM. Closed on Mondays.",
        metadata={"source": "hours.txt"},
    ),
    Document(
        page_content="Cafe Lumiere features a cozy atmosphere with jazz playlists and rotating art exhibits from local artists.",
        metadata={"source": "ambience.txt"},
    ),
]


## Step 3: Vector Store Setup

We embed our documents using HuggingFace embeddings and store them in a local vector store (Chroma).

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding_function = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# Create vector DB from documents
db = Chroma.from_documents(docs, embedding_function)
retriever = db.as_retriever(search_kwargs={"k": 2})  # Retrieve top 2 relevant docs


## Step 4: Define RAG Prompt and LLM

In [ ]:
from langchain import hub
from langchain_google_genai import ChatGoogleGenerativeAI

# Load pre-defined RAG prompt template
prompt = hub.pull("rlm/rag-prompt")
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash-exp")

# Helper to join docs into a single context string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Combine prompt and LLM
rag_chain = prompt | llm

## Step 5: Build Classifier Node

This node determines whether a user's question is related to Cafe Lumiere.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

# Define expected structured output
class GradeQuestion(BaseModel):
    score: str = Field(description="If question is about Cafe Lumiere info, respond with 'Yes', else 'No'")

def question_classifier(state):
    question = state["messages"][-1].content

    system = """You are a classifier for questions about Cafe Lumiere's founder, menu pricing, or opening hours."""

    grade_prompt = ChatPromptTemplate.from_messages([
        ("system", system),
        ("human", "User question: {question}")
    ])

    structured_llm = llm.with_structured_output(GradeQuestion)
    classifier_chain = grade_prompt | structured_llm
    result = classifier_chain.invoke({"question": question})
    state["on_topic"] = result.score
    return state


## Step 6: Define RAG and Response Nodes

In [ ]:
from langchain_core.messages import AIMessage

# Retrieve documents based on user question
def retrieve(state):
    question = state["messages"][-1].content
    state["documents"] = retriever.invoke(question)
    return state

# Generate LLM response from context and question
def generate_answer(state):
    question = state["messages"][-1].content
    documents = state["documents"]
    response = rag_chain.invoke({"context": documents, "question": question})
    state["messages"].append(response)
    return state

# Return fallback message for off-topic questions
def off_topic_response(state):
    state["messages"].append(AIMessage(content="Sorry, I can only help with questions about Cafe Lumiere."))
    return state


## Step 7: Define LangGraph Agent Workflow

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

class AgentState(TypedDict):
    messages: list
    documents: list
    on_topic: str

workflow = StateGraph(AgentState)
workflow.add_node("topic_decision", question_classifier)
workflow.add_node("retrieve", retrieve)
workflow.add_node("generate_answer", generate_answer)
workflow.add_node("off_topic_response", off_topic_response)

# Conditional routing based on classifier output
def route(state):
    return "on_topic" if state["on_topic"].lower() == "yes" else "off_topic"

workflow.add_conditional_edges("topic_decision", route, {
    "on_topic": "retrieve",
    "off_topic": "off_topic_response"
})

workflow.add_edge("retrieve", "generate_answer")
workflow.add_edge("generate_answer", END)
workflow.add_edge("off_topic_response", END)
workflow.set_entry_point("topic_decision")

graph = workflow.compile()


In [ ]:
from IPython.display import Image, display
from langchain_core.runnables.graph import MermaidDrawMethod

display(
    Image(
        graph.get_graph().draw_mermaid_png(
            draw_method=MermaidDrawMethod.API,
        )
    )
)

Invoke the agent:

In [ ]:
from langchain_core.messages import HumanMessage

graph.invoke({
    "messages": [HumanMessage(content="What is the price of espresso drinks?")]
})


In [ ]:
graph.invoke({
    "messages": [HumanMessage(content="Do you know about Virat Kohli?")]
})


## Step 8: Define RAG as a Tool

In [ ]:
from langchain.tools.retriever import create_retriever_tool
from langchain_core.tools import tool

retriever_tool = create_retriever_tool(
    retriever,
    "retriever_tool",
    "Information related to pricing, hours, or the founder of Cafe Lumiere."
)

@tool
def off_topic():
    """For unrelated questions."""
    return "Forbidden - do not respond."


Define agent tool workflow:

In [ ]:
from typing import Annotated, Sequence, Literal
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class ToolAgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

# Define agent behavior with bound tools
def agent(state):
    model = ChatGoogleGenerativeAI(model="gemini-2.0-flash-exp").bind_tools([retriever_tool, off_topic])
    response = model.invoke(state["messages"])
    return {"messages": [response]}

def should_continue(state) -> Literal["tools", END]:
    messages = state["messages"]
    last_message = messages[-1]
    if last_message.tool_calls:
        return "tools"
    return END


Final tool-based graph:

In [ ]:
from langgraph.graph import StateGraph, START
from langgraph.prebuilt import ToolNode

workflow = StateGraph(ToolAgentState)
workflow.add_node("agent", agent)
tool_node = ToolNode([retriever_tool, off_topic])
workflow.add_node("tools", tool_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue)
workflow.add_edge("tools", "agent")

tool_graph = workflow.compile()


In [ ]:
display(
    Image(
        tool_graph.get_graph().draw_mermaid_png(
            draw_method=MermaidDrawMethod.API,
        )
    )
)

Invoke the tool agent:

In [ ]:
from langchain_core.messages import HumanMessage

tool_graph.invoke({
    "messages": [HumanMessage(content="How's the weather today?")]
})


In [ ]:
from langchain_core.messages import HumanMessage
inputs = {"messages": [HumanMessage(content="Tell me about Cafe Lumiere."), HumanMessage(content="Do you know chiba prefecture?")]}

for state in tool_graph.stream(inputs, stream_mode="values"):
    last_message = state["messages"][-1]
    last_message.pretty_print()